# Part 1: Exploring the Game Data

In [ ]:
%load_ext autoreload
%autoreload 2

### Imports

In [ ]:
import sys
from nba_api.stats.endpoints import leaguegamefinder
import duckdb

## Opening the Data

In [ ]:
finder = leaguegamefinder.LeagueGameFinder(
    league_id_nullable="00",
    season_type_nullable="Regular Season",
    date_from_nullable="10/20/2026",
    date_to_nullable="10/20/2026",
)

In [ ]:
df = finder.get_data_frames()[0]

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df[["GAME_ID","SEASON_ID", "GAME_DATE", "TEAM_ABBREVIATION", "MATCHUP", "WL", "PTS"]]

In [ ]:
df.head(4)

In [ ]:
df.dtypes

First Check results:
- 6 rows from first three games due to 2 teams a game.
- No home and away columns which could leave behind vital information, which can be extracted from the matchup string data.
- No tracked W-L record or last 5 games W-L, which could also be useful.
- Game date is stored as a string rather than as a date
- Team_id stored as an integer instead of a string

### Table Creation

In [ ]:
con = duckdb.connect("../data/nba.duckdb")

In [ ]:
con.execute("""
            CREATE TABLE IF NOT EXISTS raw_team_games(
                SEASON_ID   VARCHAR,
                TEAM_ID     VARCHAR,
                TEAM_ABBREVIATION VARCHAR,
                TEAM_NAME   VARCHAR,
                GAME_ID     VARCHAR,
                GAME_DATE   DATE,
                MATCHUP     VARCHAR,
                WL          VARCHAR,
                MIN         INTEGER,
                PTS         INTEGER,
                FGM         INTEGER,
                FGA         INTEGER,
                FG_PCT      DOUBLE,
                FG3M        INTEGER,
                FG3A        INTEGER,
                FG3_PCT     DOUBLE,
                FTM         INTEGER,
                FTA         INTEGER,
                FT_PCT      DOUBLE,
                OREB        INTEGER,
                DREB        INTEGER,
                REB         INTEGER,
                AST         INTEGER,
                STL         INTEGER,
                BLK         INTEGER,
                TOV         INTEGER,
                PF          INTEGER,
                PLUS_MINUS  INTEGER,
                INGESTED_AT TIMESTAMP,
                PRIMARY KEY (GAME_ID, TEAM_ID)
            )
            """)

In [ ]:
con.sql("DESCRIBE raw_team_games")

In [ ]:
con.execute("""
            CREATE TABLE IF NOT EXISTS players (
                player_id   VARCHAR,
                name    VARCHAR,
                height_cm   INTEGER,
                fg_pct  DOUBLE,
                birth_date  DATE,
                loaded_at   TIMESTAMP,
                PRIMARY KEY (player_id)
            );
            """)

In [ ]:
con.sql("DESCRIBE players")

### Games first test

In [ ]:
from nba_predictor_v2.ingest.games import fetch_games, create_table, load_games


In [ ]:
test = fetch_games("10/20/2026")
test.shape

In [ ]:
con = duckdb.connect("../data/nba.duckdb")
create_table(con)

In [ ]:
df = fetch_games("10/20/2026")
load_games(con, df)
con.sql("SELECT COUNT(*) FROM raw_team_games")